# Synchrony 원본 로드와 기본 전처리

고객·거래 데이터를 불러와 날짜와 식별자를 정리하고, 데이터 사전·결측치·코드 연결·관측 기간을 확인합니다.
ABC 카드 보유 고객·이용·해지 전 행동은 [02_eda.ipynb](02_eda.ipynb)에서 탐색합니다.
두 노트북은 동일한 원본과 `read_source`를 사용하며 각각 독립적으로 실행할 수 있습니다.

**작업 전제:** 제공 기록에 수집 누락이 없고 거래·해지 정보는 기록된 날짜에 이용 가능합니다.
관측 기간 안의 거래 기록 부재는 거래 없음으로 처리합니다.

## 1. 경로 설정

원본 ZIP의 기본 위치는 저장소의 `data/raw/Synchrony/Datasets.zip`입니다.
다른 파일을 사용하려면 `SOURCE_ZIP` 또는 `SYNCHRONY_SOURCE_ZIP` 환경 변수를 변경합니다.

In [1]:
from pathlib import Path
import os
import sys
import zipfile

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/experiments/synchrony/data.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Cardops 저장소 안에서 노트북을 실행해 주세요.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE_ZIP = Path(os.environ.get(
    "SYNCHRONY_SOURCE_ZIP", PROJECT_ROOT / "data/raw/Synchrony/Datasets.zip"
)).expanduser()
PROCESSED_DIR = PROJECT_ROOT / "data/processed/Synchrony"

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f"원본 ZIP을 준비하거나 SOURCE_ZIP을 변경해 주세요: {SOURCE_ZIP}")

print("저장소:", PROJECT_ROOT)
print("원본:", SOURCE_ZIP)
print("향후 정제본 저장 위치:", PROCESSED_DIR)

저장소: /Users/geonwookim/Desktop/skn34_project/Cardops
원본: /Users/geonwookim/Desktop/skn34_project/Cardops/data/raw/Synchrony/Datasets.zip
향후 정제본 저장 위치: /Users/geonwookim/Desktop/skn34_project/Cardops/data/processed/Synchrony


## 2. 원본 로드와 날짜 변환

기존 실험의 `read_source`를 재사용합니다. 발급일·해지일·거래일을 날짜형으로 변환하며,
고객 ID와 거래 ID의 중복, 거래의 고객 ID가 고객 테이블에 존재하는지 검사합니다.

해지일이 비어 있다는 이유로 고객을 삭제하거나 임의의 날짜를 채우지 않습니다.

In [2]:
from src.experiments.synchrony.data import read_source

customers, transactions = read_source(SOURCE_ZIP)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    payment_codes = pd.read_csv(archive.open("Datasets/Payment Code.csv"))
    category_codes = pd.read_csv(archive.open("Datasets/Category Code.csv"))

display(pd.DataFrame([
    {"테이블": "고객", "행 수": len(customers), "열 수": len(customers.reset_index().columns)},
    {"테이블": "거래", "행 수": len(transactions), "열 수": len(transactions.columns)},
    {"테이블": "결제수단", "행 수": len(payment_codes), "열 수": len(payment_codes.columns)},
    {"테이블": "상품 범주", "행 수": len(category_codes), "열 수": len(category_codes.columns)},
]))
# 거래 테이블에는 read_source가 거래일에서 만든 month 열이 추가됩니다.
display(customers.head(), transactions.head())

,테이블,행 수,열 수
0,고객,45000,8
1,거래,446425,9
2,결제수단,5,2
3,상품 범주,10,2


,Age,Gender,Membership_Type,Credit_Card_Open_Date,Credit_Card_Closed_Date,Credit_Card_Limit,Credit_Card_APR
Customer_ID,,,,,,,
99610,46,Male,Prime,2023-10-03,NaT,175000.0,30.31
22456,39,Male,Prime,2024-12-15,NaT,290000.0,33.72
63403,18,Male,Non-Prime,2022-06-07,NaT,246000.0,34.96
72905,51,Female,Non-Prime,2024-05-01,NaT,220000.0,33.89
76339,27,Male,Non-Prime,2022-07-23,NaT,131000.0,30.42


,Customer_ID,Transaction_ID,Transaction_Date,Category_Code,Transaction_Type,Transaction_Amount,Payment_Code,Number_of_Transactions,month
0,99610,5921081,2025-09-28,1,Sale,853.82,4,10.0,2025-09
1,99610,3241432,2026-03-20,8,Sale,1943.64,3,10.0,2026-03
2,99610,7477582,2026-05-23,3,Sale,1496.84,2,10.0,2026-05
3,99610,5359430,2024-10-07,5,Sale,5473.40,3,10.0,2024-10
4,99610,5201054,2024-09-16,2,Sale,2796.16,1,10.0,2024-09


### 데이터 사전 (Data Dictionary)

아래 타입은 `read_source`와 `pd.read_csv`로 로드한 뒤의 pandas 자료형입니다. 원본 ZIP의 네 CSV와 현재 전처리 코드를 기준으로 정리했습니다.

#### 고객 데이터 — `Customer Data.csv` → `customers`

| 컬럼명 | 설명 | 타입 | 비고 |
| --- | --- | --- | --- |
| `Customer_ID` | 고객 고유 식별번호 | 식별자 (`int64`) | 로드 후 `customers`의 인덱스가 되며 거래 테이블과 연결하는 키 |
| `Age` | 고객 나이 | 수치형 (`int64`) | 현재 원본의 관측 범위는 18~60 |
| `Gender` | 고객 성별 | 범주형 (`object`) | `Male`, `Female` |
| `Membership_Type` | 멤버십 구분 | 범주형 (`object`) | `Prime`, `Non-Prime` |
| `Credit_Card_Open_Date` | ABC 카드 발급일 | 날짜 (`datetime64[ns]`) | 원본 `월/일/연도` 문자열을 변환; 발급 기록이 있는 고객의 기준일별 보유 여부 판단에 사용 |
| `Credit_Card_Closed_Date` | 기록된 ABC 카드 해지일 | 날짜 (`datetime64[ns]`) | `NaT`는 해지 기록 없음; 발급일도 함께 확인해야 하며, 이후 30일·60일 해지 정답을 만들 때 사용 |
| `Credit_Card_Limit` | ABC 카드 신용한도 | 수치형 (`float64`) | 결측 유지; 통화 단위는 원본 CSV에 명시되지 않음 |
| `Credit_Card_APR` | ABC 카드 연이율(APR) 값 | 수치형 (`float64`) | 결측 유지; 구체적인 표기 단위는 원본 CSV에 명시되지 않음 |

현재 원본에서 발급일·한도·APR는 같은 **6,836명**에게 함께 비어 있습니다. 이는 카드 정보가 기록되지 않은 상태이며, 미발급 여부를 임의로 확정하거나 0으로 대체하지 않습니다. 해지일이 비어 있는 **37,336명**에는 이 고객들도 포함되므로 해지일 결측만으로 카드 보유 중이라고 판단하지 않습니다.

#### 거래 데이터 — `Transaction Data.csv` → `transactions`

| 컬럼명 | 설명 | 타입 | 비고 |
| --- | --- | --- | --- |
| `Customer_ID` | 거래 기록의 고객 식별번호 | 식별자 (`int64`) | `customers` 인덱스와 연결; 존재하지 않는 고객을 참조하면 로드 단계에서 오류 발생 |
| `Transaction_ID` | 거래 기록 고유 식별번호 | 식별자 (`int64`) | 중복 여부 검사에 사용; 한 기록에 여러 거래가 집계될 수 있어 행 수와 거래건수는 구분 |
| `Transaction_Date` | 거래일 | 날짜 (`datetime64[ns]`) | 원본 `월/일/연도` 문자열을 변환; 현재 관측 범위는 2024-08-01~2026-07-31 |
| `Category_Code` | 상품 범주 코드 | 범주형 코드 (`int64`) | `Category Code.csv`의 `Category_Code`와 연결; 숫자의 크기를 순서로 해석하지 않음 |
| `Transaction_Type` | 구매·반품 구분 | 범주형 (`object`) | `Sale` = 구매, `Return` = 반품; 구매와 반품을 분리해 집계 |
| `Transaction_Amount` | 거래 기록에 기재된 금액 | 수치형 (`float64`) | 원본의 반품 금액도 양수; 현재 코드는 기록 금액을 합산하며 통화 단위는 CSV에 명시되지 않음 |
| `Payment_Code` | 결제수단 코드 | 범주형 코드 (`int64`) | `Payment Code.csv`와 연결; 현재 대상 카드는 `3` (`ABC Bank Credit Card`) |
| `Number_of_Transactions` | 해당 기록에 집계된 거래건수 | 수치형 (`float64`) | 구매 건수는 `Sale` 행의 값을 합산; 현재 원본의 반품 44,449행은 모두 결측이므로 임의로 0이나 1을 채우지 않음 |
| `month` | 거래일에서 만든 거래 연월 | 월 기간 (`period[M]`) | **파생 컬럼**: `read_source`가 `Transaction_Date.dt.to_period("M")`로 생성; 원본 CSV에는 없음 |

#### 결제수단 코드표 — `Payment Code.csv` → `payment_codes`

`Payment_Code`는 결제수단 코드(`int64`), `Payment_Method`는 결제수단 이름(`object`)입니다.

| `Payment_Code` | `Payment_Method` (원본) | 설명 |
| --- | --- | --- |
| 1 | Debit Card | 직불카드 |
| 2 | Other Bank Credit Card | 타 은행 신용카드 |
| 3 | ABC Bank Credit Card | 현재 분석 대상 신용카드 |
| 4 | Cash/UPI | 현금 또는 UPI 결제 |
| 5 | XYZ Wallet | XYZ 지갑 결제 |

#### 상품 범주 코드표 — `Category Code.csv` → `category_codes`

`Category_Code`는 상품 범주 코드(`int64`), `Category`는 상품 범주 이름(`object`)입니다. 아래 영문 값은 원본 표기를 유지합니다.

| `Category_Code` | `Category` (원본) | 설명 |
| --- | --- | --- |
| 1 | Grocery | 식료품 |
| 2 | Electronics | 전자제품 |
| 3 | Apparel | 의류 |
| 4 | Beauty | 뷰티 |
| 5 | Furniture | 가구 |
| 6 | Kids And Toys | 아동용품·장난감 |
| 7 | Large Applicances | 대형 가전 (원본 철자 유지) |
| 8 | Outdoor | 아웃도어 |
| 9 | Travel | 여행 |
| 10 | Bill Payments | 청구요금 납부 |

## 3. 자료형과 결측치 확인

결측치를 일괄 삭제·대체하기 전에 각 열의 의미를 확인합니다.
고객의 발급일이 없는 경우와 해지일이 없는 경우는 별도로 해석합니다.

In [3]:
def column_summary(frame):
    return pd.DataFrame({
        "자료형": frame.dtypes.astype(str),
        "결측 수": frame.isna().sum(),
        "결측 비율(%)": frame.isna().mean().mul(100).round(2),
    })

display(column_summary(customers.reset_index()))
display(column_summary(transactions))
display(payment_codes, category_codes)

,자료형,결측 수,결측 비율(%)
Customer_ID,int64,0,0.00
Age,int64,0,0.00
Gender,object,0,0.00
Membership_Type,object,0,0.00
Credit_Card_Open_Date,datetime64[ns],6836,15.19
Credit_Card_Closed_Date,datetime64[ns],37336,82.97
Credit_Card_Limit,float64,6836,15.19
Credit_Card_APR,float64,6836,15.19


,자료형,결측 수,결측 비율(%)
Customer_ID,int64,0,0.00
Transaction_ID,int64,0,0.00
Transaction_Date,datetime64[ns],0,0.00
Category_Code,int64,0,0.00
Transaction_Type,object,0,0.00
Transaction_Amount,float64,0,0.00
Payment_Code,int64,0,0.00
Number_of_Transactions,float64,44449,9.96
month,period[M],0,0.00


,Payment_Code,Payment_Method
0,1,Debit Card
1,2,Other Bank Credit Card
2,3,ABC Bank Credit Card
3,4,Cash/UPI
4,5,XYZ Wallet


,Category_Code,Category
0,1,Grocery
1,2,Electronics
2,3,Apparel
3,4,Beauty
4,5,Furniture
5,6,Kids And Toys
6,7,Large Applicances
7,8,Outdoor
8,9,Travel
9,10,Bill Payments


### 결측치 점검 결과와 처리 기준

현재 로컬 원본에는 **고객 데이터 4개 컬럼, 거래 데이터 1개 컬럼에 결측치**가 있습니다. 아래 수치는 고객 **45,000명**, 거래 **446,425행**을 기준으로 확인했습니다. 원본을 교체하면 위 요약 셀을 다시 실행해 수치를 확인합니다.

| 테이블 | 컬럼 | 결측 수 | 해당 테이블 내 비율 |
| --- | --- | ---: | ---: |
| 고객 | `Credit_Card_Open_Date` — 카드 발급일 | 6,836명 | 15.19% |
| 고객 | `Credit_Card_Closed_Date` — 카드 해지일 | 37,336명 | 82.97% |
| 고객 | `Credit_Card_Limit` — 카드 한도 | 6,836명 | 15.19% |
| 고객 | `Credit_Card_APR` — 카드 연이율 | 6,836명 | 15.19% |
| 거래 | `Number_of_Transactions` — 거래건수 | 44,449행 | 9.96% |

**현재 처리:** `read_source`는 원본 결측을 삭제하거나 대체하지 않습니다. 날짜를 변환한 뒤 날짜 결측은 `NaT`, 수치 결측은 `NaN`으로 유지합니다. 이 노트북에서는 고객이나 거래 전체에 `dropna()` 또는 `fillna(0)`를 적용하지 않습니다.

#### 1. 발급일·한도·APR가 함께 없는 고객

- 세 컬럼은 **같은 6,836명**에게 함께 비어 있습니다. 이 고객들에게는 대상 카드(`Payment_Code = 3`) 거래도 없습니다.
- 카드 정보가 기록되지 않은 패턴이며, 원본만으로 미발급·미보유 여부를 확정할 수는 없습니다. 발급일을 임의 날짜로 채우거나 한도·APR를 0으로 바꾸지 않습니다.
- 이 노트북에서는 해당 고객을 원본에 유지합니다. 별도의 [월별 스냅샷 코드](../../src/experiments/synchrony/data.py)는 기준일까지 발급되었고 아직 해지되지 않은 고객을 선택하므로, 발급일이 없는 고객은 예측 대상에서 제외됩니다.

#### 2. 해지일이 없는 고객

- 해지일 결측 **37,336명**은 **발급 기록은 있으나 해지 기록은 없는 30,500명**과 **발급일도 없는 6,836명**으로 구성됩니다.
- 해지일 결측은 **관측 기록에 해지일이 없음**을 뜻합니다. 카드 보유 여부는 발급일과 함께 확인하고, 미래에도 해지하지 않을 고객으로 확정하지 않습니다.
- 고객 테이블에 일괄 `dropna()`를 적용하면 해지일이 기록된 **7,664명만 남아** 분석 대상이 치우칩니다. 따라서 해지일 결측을 이유로 고객을 삭제하거나 임의의 해지일을 채우지 않습니다.

#### 3. 반품 기록의 거래건수 결측

- 구매(`Sale`) **401,976행**은 거래건수 결측이 없고, 반품(`Return`) **44,449행**은 거래건수가 전부 결측입니다.
- [02 EDA](02_eda.ipynb)는 `Sale`만 선택해 `Number_of_Transactions`를 합산합니다. 반품의 결측이 구매 거래건수 집계에 섞이지 않습니다.
- 반품 거래건수를 임의로 0이나 1로 채우지 않습니다. 반품은 기록된 금액과 **기록 행 수**로 별도 확인하며, 행 수를 실제 반품 건수와 동일하게 해석하지 않습니다.

결측값을 채우거나 제외하는 규칙을 추가할 때는 위 패턴과 분석 목적을 먼저 확인합니다. 01에서는 원본 결측을 보존하고, [02 EDA](02_eda.ipynb)에서 구매·반품을 구분해 집계합니다.

## 4. 거래 관측 기간 확인

거래일의 시작·종료와 결측 여부를 확인합니다.
월별 구매 규모와 이용 추이는 [02 EDA의 월별 분석](02_eda.ipynb)에서 확인합니다.

In [4]:
observation_start = transactions["Transaction_Date"].min()
observation_end = transactions["Transaction_Date"].max()
assert transactions["Transaction_Date"].notna().all(), "거래일 결측을 확인해 주세요."
print(f"거래 관측 기간: {observation_start.date()} ~ {observation_end.date()}")
print(f"거래 기록이 있는 월 수: {transactions['month'].nunique()}개월")

거래 관측 기간: 2024-08-01 ~ 2026-07-31
거래 기록이 있는 월 수: 24개월


## 5. 결제수단·상품 범주 코드 확인

거래의 코드가 원본 코드표에 존재하는지 확인하고, 대상 카드인 `Payment_Code = 3`의 이름을 확인합니다.
대상 카드의 구매 고객 수·금액·거래건수와 이용 비중은 [02 EDA](02_eda.ipynb)에서 집계합니다.

In [5]:
CARD_PAYMENT_CODE = 3
code_checks = pd.Series({
    "결제수단 코드표 키 고유": payment_codes["Payment_Code"].is_unique,
    "상품 범주 코드표 키 고유": category_codes["Category_Code"].is_unique,
    "모든 결제수단 코드 연결": transactions["Payment_Code"].isin(payment_codes["Payment_Code"]).all(),
    "모든 상품 범주 코드 연결": transactions["Category_Code"].isin(category_codes["Category_Code"]).all(),
    "대상 카드 코드 존재": payment_codes["Payment_Code"].eq(CARD_PAYMENT_CODE).any(),
}, name="검사 통과")
display(code_checks)
assert code_checks.all(), "연결되지 않은 코드 또는 중복 코드표 키를 확인해 주세요."
display(payment_codes.loc[payment_codes["Payment_Code"].eq(CARD_PAYMENT_CODE)])

결제수단 코드표 키 고유     True
상품 범주 코드표 키 고유    True
모든 결제수단 코드 연결     True
모든 상품 범주 코드 연결    True
대상 카드 코드 존재       True
Name: 검사 통과, dtype: bool

,Payment_Code,Payment_Method
2,3,ABC Bank Credit Card


## 6. 다음 단계: [02_eda.ipynb](02_eda.ipynb)

원본 로드와 기본 점검을 마쳤습니다. [02 ABC 카드 EDA](02_eda.ipynb)에서 다음 항목을 탐색합니다.

- ABC 카드 보유 고객과 발급·해지 상태: ABC 구매 0건 고객도 포함
- ABC 구매·반품, 월별 사용·발급·해지, ABC 구매 상품 범주
- 현재 보유 고객의 최근 90일 구매·미사용·사용 감소
- 같은 기준일의 ABC 고객에게서 과거 행동과 이후 60일 해지 기록 비교
- 동일 ABC 고객·동일 기간의 타 결제수단 사용을 보조 분석

02는 원본을 다시 로드하므로 이 노트북의 실행 변수에 의존하지 않습니다.
02의 해지 비교는 탐색용이며, 모델용 자료 저장·여러 기준일의 특징과 정답 생성·학습 및 평가 기간 분할은 다음 단계로 이어집니다.